<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 2 (AI): Parameters, Advanced Prompting, Structured Outputs & Tools

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Learn

In this notebook, you will:

1. **Set the model's knobs** — temperature, max_tokens, top_p, penalties
2. **Stream** a reply so it types out live
3. **Teach by example** (few-shot) and trigger **step-by-step reasoning**
4. **Validate data with Pydantic** — and how it differs from `TypedDict`
5. Get a **guaranteed output shape** with `parse()`
6. **Give the model a tool**, one clear step at a time (function calling)

---

## 1. Environment Setup

Run these first. You'll need an **OpenAI API key**; Gemini is optional (press Enter to skip).

In [ ]:
# Install the packages we need
!pip install -q openai litellm pydantic

In [ ]:
# Imports
import os
import json
from enum import Enum
from getpass import getpass
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError

In [ ]:
# API keys (typed securely - not shown on screen)
openai_api_key = getpass("Enter your OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = openai_api_key

# Gemini is optional - press Enter to skip
google_api_key = getpass("Enter your Gemini API Key (or press Enter to skip): ")

# Models we'll use (change to any you have access to)
OPENAI_MODEL = "gpt-4o-mini"
GEMINI_MODEL = "gemini-2.0-flash"

client = OpenAI()  # reads OPENAI_API_KEY from the environment
print("Ready. OpenAI model:", OPENAI_MODEL)

## 2. LLM Call Parameters — the knobs

Every `create(...)` call takes optional knobs that shape the output. You met **temperature** already; here are the common ones you'll actually use.

| Knob | What it does |
|---|---|
| `temperature` | randomness: 0 = focused & repeatable, high = creative |
| `top_p` | nucleus sampling - another randomness dial (tune this *or* temperature) |
| `max_tokens` | caps the reply **length** (newer name: `max_completion_tokens`) |
| `frequency_penalty` | less repetition of the **same words** |
| `presence_penalty` | pushes toward **new topics** |
| `top_k` | top-k sampling - **not on OpenAI**; runs on **Gemini** (below) |

*(Other handy ones: `stop` = stop sequences, `seed` = more reproducible output.)*

In [ ]:
# temperature = randomness. Change it and re-run (try 0, then 1.2).
temperature = 0.7

r = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": "Give me a tagline for a chai shop."}],
    temperature=temperature,
)
print(r.choices[0].message.content)

**max_tokens** caps the reply length. Set it tiny and watch the answer get cut off.

In [ ]:
# (Newer / reasoning models call this `max_completion_tokens` - same idea.)
r = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": "Explain how the internet works."}],
    max_tokens=20,
)
print(r.choices[0].message.content)
print("\nstopped because:", r.choices[0].finish_reason)   # 'length' = it hit the cap

**top_p** (nucleus sampling) is another randomness dial. Rule of thumb: tune **temperature OR top_p**, not both.

In [ ]:
# Low top_p = play safe (top few words); 1.0 = open to everything. Change & re-run.
top_p = 0.2

r = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": "Invent a name for a new planet."}],
    top_p=top_p,
)
print(r.choices[0].message.content)

**Penalties** cut repetition — both range **-2.0 to 2.0**. They sound alike but do different jobs, so take them one at a time.

### frequency_penalty — stop reusing the *same words*

`frequency_penalty` looks at **how often** a word has already appeared and discourages it **more each time it repeats**. Higher = more varied wording.

**Analogy:** a teacher saying *"you've written 'amazing' five times - reach for other words."*

In [ ]:
# Higher = avoid repeating the SAME words. Change frequency_penalty and re-run (0 vs 1.8).
frequency_penalty = 0.0

r = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": "In 4 sentences, tell me why you love pizza."}],
    frequency_penalty=frequency_penalty,
)
print(r.choices[0].message.content)
# At 0 it repeats "pizza"/"love"; at 1.8 it reaches for synonyms and different words.

### presence_penalty — move on to *new topics*

`presence_penalty` checks whether a word has appeared **at all** (a flat, one-time nudge) and pushes the model toward **new topics**, not just new words. Higher = broader coverage.

**Analogy:** *"you already talked about the taste - now mention price, toppings, memories..."*

In [ ]:
# Higher = bring in NEW topics. Change presence_penalty and re-run (0 vs 1.8).
presence_penalty = 0.0

r = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": "Give me ideas for a birthday party."}],
    presence_penalty=presence_penalty,
)
print(r.choices[0].message.content)
# At 0 the ideas cluster (cake, balloons); at 1.8 they spread to new themes and activities.

> **In one line:** `frequency_penalty` = don't repeat the same **word**; `presence_penalty` = move on to a new **topic**.

### top_k — reach Gemini with LiteLLM

`top_k` keeps only the **k most likely** next tokens (`top_k=1` = greedy / most focused; higher = more variety) - another randomness dial like `top_p`.

⚠️ **OpenAI's API has no `top_k`.** But **LiteLLM** (Day 1's one-interface trick) can forward it to a provider that *does* - here **Gemini**. (Needs your Gemini key.)

In [ ]:
# top_k isn't an OpenAI param - LiteLLM forwards it to Gemini. Change top_k and re-run (1 = focused, 40 = varied).
from litellm import completion

if google_api_key:
    os.environ["GEMINI_API_KEY"] = google_api_key
    r = completion(
        model="gemini/gemini-2.0-flash",
        messages=[{"role": "user", "content": "Invent 3 unusual names for a coffee shop."}],
        top_k=1,
    )
    print(r.choices[0].message.content)
else:
    print("No Gemini key - skipping. (top_k is Gemini/Claude only, not OpenAI.)")

## ⚡ Streaming — see the answer as it's written

By default you wait for the whole reply, then it appears at once. With **streaming** the model sends the answer **token by token**, so the user watches it type out (like ChatGPT). Set `stream=True` and loop over the chunks - here via **LiteLLM**.

In [ ]:
# stream=True -> tokens arrive as they're generated. Print each piece as it comes in.
from litellm import completion

stream = completion(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": "Tell me a short story about a robot in 4 sentences."}],
    stream=True,
)
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="", flush=True)   # or "" guards empty chunks

A chat UI (like the **Gradio** companion notebook) wants a function that **`yield`s the answer-so-far**. Same idea, wrapped in a generator:

In [ ]:
def stream_answer(prompt, system_message="You are a helpful assistant."):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt},
    ]
    stream = completion(model="openai/gpt-4o-mini", messages=messages, stream=True)

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result                 # each yield gives the growing text

# watch it grow (a chat UI renders each update live):
from IPython.display import clear_output
for partial in stream_answer("Give me one fun fact about space."):
    clear_output(wait=True)
    print(partial)

## 3. Zero-shot vs Few-shot (teaching by example)

**Zero-shot** = instructions only, no examples.
**Few-shot** = put a few **worked examples** (input -> output) in the prompt and let the model copy the pattern. This is **in-context learning**: you teach a new task *at runtime*, no retraining.

We want tickets labeled with **our** taxonomy: `urgent` / `normal` / `spam`.

In [ ]:
# Zero-shot: instructions only. Change `ticket` and re-run to compare.
ticket = "URGENT!! payment taken twice, need my money back now"

resp = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": f"Classify this support ticket:\n{ticket}"}],
)
print(resp.choices[0].message.content)   # often a sentence or its OWN label, not our 3 labels

Now the **few-shot** version: we show three examples first, so the model answers with **exactly one** of our labels.

In [ ]:
# Few-shot: 3 examples teach the label set. Same `ticket` variable - change & re-run.
ticket = "URGENT!! payment taken twice, need my money back now"

prompt = f'''Label each support ticket as one word: urgent, normal, or spam.

Ticket: "the app crashes every time I open it"      -> urgent
Ticket: "how do I change my profile photo?"          -> normal
Ticket: "You WON a free iPhone, click here now"      -> spam
Ticket: "{ticket}" ->'''

resp = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": prompt}],
)
print(resp.choices[0].message.content)   # -> a single label from OUR set

## 4. Chain-of-Thought - give it room to think

Ask for a multi-step answer immediately and the model often **guesses wrong**. Ask it to **work through the steps** and accuracy jumps. The magic words: **"Let's think step by step."**

In [ ]:
# Answer-only: change `question` and re-run. Often wrong on multi-step math.
question = "A shop had 23 apples, sold 17, got 40 more, then sold half. How many now?"

resp = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": f"{question} Reply with only the final number."}],
)
print(resp.choices[0].message.content)

Same question, but we ask it to **reason first**. Watch it lay out the steps and land the right answer.

In [ ]:
# Chain-of-thought: the reasoning tokens are the model "working on paper".
question = "A shop had 23 apples, sold 17, got 40 more, then sold half. How many now?"

resp = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": f"{question}\nLet's think step by step, then give the final number."}],
)
print(resp.choices[0].message.content)

> **Reasoning models (2026):** newer models (GPT-5 *thinking* / o-series, Gemini *thinking*, Claude *extended thinking*) do this **internally** and expose a *reasoning-effort* knob. On those you usually **don't** paste "think step by step" - they already do. Use chain-of-thought where correctness matters; skip it for simple lookups (it just costs tokens).

## 5. Pydantic - the shape *and* the checker

**Pydantic** lets you declare the *shape* of data and it **checks it at runtime**. You give each field a type (and optional rules); good data passes, bad data is rejected with a clear error. This is what makes LLM output safe for code (next section) - and it's the same tool FastAPI uses to validate requests.

In [ ]:
# Declare a shape with rules, then build a VALID object.
class Student(BaseModel):
    name: str = Field(min_length=1)      # cannot be empty
    age: int = Field(ge=15, le=100)      # must be 15-100
    interests: list[str] = []            # optional, defaults to empty

good = Student(name="Ada", age=20, interests=["ai", "math"])
print(good)

Now feed it **bad** data. Pydantic doesn't just hint - it **raises** and tells you exactly what's wrong.

In [ ]:
# validation in action: empty name + age out of range
try:
    Student(name="", age=200)
except ValidationError as e:
    print(e)

### Pydantic vs `TypedDict`

A `TypedDict` looks similar, but it's **only a hint** for editors / type-checkers - at runtime it does **no checking** at all.

In [ ]:
from typing import TypedDict

class StudentTD(TypedDict):
    name: str
    age: int

# age should be an int... but TypedDict never checks. This runs happily:
s = StudentTD(name="Ada", age="ten")
print(s, "   <- no error! TypedDict is just a plain dict at runtime")

# Pydantic, by contrast, would REJECT age="ten" (try it):
# Student(name="Ada", age="ten")   # -> ValidationError

## 6. Structured Output - a *guaranteed* shape with `parse()`

Asking for JSON only makes text *look* right; the model can still drop a field or use the wrong type. The upgrade: pass a **Pydantic model** as `response_format` and call **`parse()`**. You get a **typed object** back, shape guaranteed - Pydantic (section 5) is doing the validation under the hood.

In [ ]:
# Note: .parse() (not .create). Pass the class; read .parsed - already typed.
class Recipe(BaseModel):
    title: str
    ingredients: list[str]
    minutes: int

completion = client.chat.completions.parse(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": "A simple pasta recipe."}],
    response_format=Recipe,
)

recipe = completion.choices[0].message.parsed   # a real Recipe object
print("Title:  ", recipe.title)
print("Minutes:", recipe.minutes)
print("Items:  ", ", ".join(recipe.ingredients))

An **`Enum`** locks the model to *your* categories - section 3's classifier, now **enforced** (it literally cannot invent a 4th label).

In [ ]:
# The model can ONLY choose one of the enum values.
class Priority(str, Enum):
    urgent = "urgent"
    normal = "normal"
    spam = "spam"

class Ticket(BaseModel):
    customer: str
    priority: Priority
    summary: str

messy = "Hi it's Ravi - you charged my card TWICE this morning, fix it ASAP!!"

completion = client.chat.completions.parse(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": f"Extract a support ticket from:\n{messy}"}],
    response_format=Ticket,
)

ticket = completion.choices[0].message.parsed
print(ticket.customer, "|", ticket.priority.value, "|", ticket.summary)

## 7. Function Calling - one step at a time

An LLM only predicts text - it can't do **exact arithmetic** (it guesses). So we give it a **tool**: a plain function. The model doesn't run it - it **asks us to**, and **our** code runs it. That boundary is the safety story.

We'll build it in **5 small steps** with one simple tool: a calculator.

**Step 1 - write a normal Python function.** No AI yet. Just a function that does exact math.

In [ ]:
def calculator(a: float, b: float, op: str) -> dict:
    result = {"add": a + b, "sub": a - b, "mul": a * b,
              "div": a / b if b else None}[op]
    return {"result": result}

print(calculator(12, 7, "add"))   # {'result': 19}   <- plain Python, works

**Step 2 - describe the tool to the model.** This schema is *all* the model knows about our function: its name, when to use it, and its arguments.

In [ ]:
tools = [{
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Do exact arithmetic on two numbers.",
        "parameters": {
            "type": "object",
            "properties": {
                "a":  {"type": "number"},
                "b":  {"type": "number"},
                "op": {"type": "string", "enum": ["add", "sub", "mul", "div"]},
            },
            "required": ["a", "b", "op"],
            "additionalProperties": False,
        },
    },
}]
print("tool described")

**Step 3 - ask a question.** With the tool available, the model replies with a **tool_call** (a request), not a normal answer.

In [ ]:
messages = [{"role": "user", "content": "What is 47853 times 1942?"}]

resp = client.chat.completions.create(model=OPENAI_MODEL, messages=messages, tools=tools)
msg = resp.choices[0].message

if msg.tool_calls:                        # real code always checks this
    call = msg.tool_calls[0]
    print("wants to call:", call.function.name)
    print("with args:    ", call.function.arguments)   # a JSON string
else:
    print("Model answered directly (re-run if you see this):", msg.content)

**Step 4 - run the function** with the arguments the model chose.

In [ ]:
args = json.loads(call.function.arguments)   # JSON string -> Python dict
result = calculator(**args)                   # actually do the math
print(args, "->", result)

**Step 5 - send the result back** so the model can answer in plain words.

In [ ]:
messages.append(msg)                          # the model's tool request
messages.append({                             # our tool's result
    "role": "tool",
    "tool_call_id": call.id,
    "content": json.dumps(result),
})

final = client.chat.completions.create(model=OPENAI_MODEL, messages=messages, tools=tools)
print(final.choices[0].message.content)       # a natural answer built from real data

> **That's the whole loop:** ask (with tools) -> the model requests a tool -> you run it and hand back the result -> the model answers. Repeated many times, this loop is the skeleton of an **agent** (Day 5), and it's how **RAG** works too (Day 4).

## 8. Exercises

Fill in the blanks (`___`) and run each cell. Use the setup and examples above.

### Q1: Few-shot classifier

Teach the model to label a movie review as `positive`, `negative`, or `mixed` using **three** examples, then classify a new review.

**Hints:** put the examples *inside* the prompt with `-> label`; end with your review and a trailing `->`.

In [ ]:
review = "The acting was superb but the plot dragged for an hour."

prompt = f'''Label each movie review as one word: positive, negative, or mixed.

Review: "Best film I have seen all year, loved every minute" -> ___
Review: "Total waste of time, I walked out halfway"          -> ___
Review: "Great visuals but a weak, confusing story"           -> ___
Review: "{review}" ->'''

resp = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "___", "content": prompt}],   # which role sends the prompt?
)
print(resp.choices[0].message.content)

### Q2: Structured output with `parse()`

Get one **book** as a typed object with fields `title`, `author`, `year`.

**Hints:** pass the class to `response_format=`; read `.parsed`.

In [ ]:
class Book(BaseModel):
    title: str
    author: str
    year: int

completion = client.chat.completions.parse(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": "Give one classic novel with its author and year."}],
    response_format=___,                 # which class guarantees the shape?
)

book = completion.choices[0].message.___ # which attribute holds the typed object?
print(book.title, "by", book.author, "(", book.year, ")")

### Q3: Function calling with a simple tool

Complete the tool call so the model can answer **"How much is a notebook?"** using `get_price`.

**Hints:** the schema `name` must match the function; the result message role is `"tool"`.

In [ ]:
def get_price(item: str) -> dict:
    catalog = {"pencil": 5, "notebook": 40, "bag": 600}
    return {"item": item, "price_inr": catalog.get(item.lower(), "unknown")}

price_tool = [{
    "type": "function",
    "function": {
        "name": "___",                   # must match the function name above
        "description": "Look up the price in INR of an item.",
        "parameters": {
            "type": "object",
            "properties": {"item": {"type": "string"}},
            "required": ["item"],
            "additionalProperties": False,
        },
    },
}]

messages = [{"role": "user", "content": "How much is a notebook?"}]
resp = client.chat.completions.create(model=OPENAI_MODEL, messages=messages, tools=price_tool)
call = resp.choices[0].message.tool_calls[0]

args = json.loads(call.function.arguments)
result = get_price(**args)

messages.append(resp.choices[0].message)
messages.append({"role": "___", "tool_call_id": call.id, "content": json.dumps(result)})  # which role?

final = client.chat.completions.create(model=OPENAI_MODEL, messages=messages, tools=price_tool)
print(final.choices[0].message.content)

---

### ✅ Recap

- **Parameters** shape every call: `temperature`/`top_p` (randomness), `max_tokens` (length), penalties (repetition); `top_k` is Gemini/Claude only.
- **Few-shot** teaches by example; **chain-of-thought** buys multi-step accuracy (reasoning models do it natively).
- **Pydantic** validates at runtime (a `TypedDict` only hints); **`parse()`** returns a guaranteed, typed object.
- **Function calling**: the model *requests* a tool, **your code runs it**, you feed the result back. That loop is the skeleton of an **agent**.